# GENEX-KT: running the open-source LLM explainer on Colab

This notebook runs Phases 3–4 of GENEX-KT (explanations and interventions) with a free
**open-source LLM on Colab's GPU**. No API key is needed. Optionally it also runs Claude for
comparison and retrains the backbone at full size.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (free tier is enough).

Run the cells top to bottom. Expect about 30–60 minutes in total for 100 students.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the project

Colab deletes files when the session ends, so the project is kept in **Google Drive**.
Upload `GENEX-KT.zip` to the top of your Drive (*My Drive*) first, then run this cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/genex_kt'
if not os.path.exists(PROJECT):
    import zipfile
    zipfile.ZipFile('/content/drive/MyDrive/GENEX-KT.zip').extractall('/content/drive/MyDrive/')
%cd {PROJECT}
!ls

## 3. Install the extra packages

In [ ]:
# PyTorch, transformers, pandas and scikit-learn are already on Colab
!pip install -q optuna krippendorff bitsandbytes accelerate anthropic

## 4. Choose the open-source model

* **LLaMA 3-8B-Instruct** — the model named in the proposal. It is *gated*: open
  https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct, accept the licence and wait for
  approval (usually minutes to hours).
* **Qwen2.5-7B-Instruct** — Apache-2.0, no approval needed. Use it if LLaMA access is pending.

Then create a *read* token at https://huggingface.co/settings/tokens and paste it when asked.

In [ ]:
MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'   # or 'Qwen/Qwen2.5-7B-Instruct'

os.environ['GENEX_HF_MODEL'] = MODEL
os.environ['GENEX_HF_4BIT'] = '1'      # 4-bit so an 8B model fits the T4 (~6 GB)

from huggingface_hub import login
login()                                # paste your Hugging Face token

## 5. Quick test on 5 students

This downloads the model once (about 5–15 minutes) and checks everything works before the
full run. Look for a faithfulness pass rate and a readable diagnosis.

In [ ]:
# test output goes to a scratch folder so it doesn't mix with the real results
!rm -rf /content/test_run && mkdir -p /content/test_run && cp -r outputs/genex /content/test_run/
!python -m genex.explain --backend hf --n 5 --out_root /content/test_run 2>&1 | grep -v -i warn | tail -3

import json
recs = [json.loads(l) for l in open('/content/test_run/explanations/explanations_hf.jsonl')]
for r in recs[:2]:
    print('=' * 80)
    print('Student', r['user'])
    print(r['text'])
    print('Faithfulness:', r['faithfulness'])

If you see `parse_error` in the output, the model did not return valid JSON. Try the other
model in step 4 and rerun. Qwen is usually the most reliable at JSON.

## 6. Full run: 100 students (Phase 3 + Phase 4)

Takes roughly 20–40 minutes on a T4. Results are written to `outputs/` in your Drive.

In [ ]:
!python -m genex.explain --backend hf --n 100 2>&1 | grep -v -i warn | tail -3
!python -m genex.intervene --backend hf 2>&1 | grep -v -i warn | tail -4

## 7. Results

In [ ]:
import pandas as pd
s = json.load(open('outputs/explanations/summary_hf.json'))
print(json.dumps(s, indent=2))
print()
print(pd.read_csv('outputs/interventions/validation_summary_hf.csv').round(3).to_string(index=False))

In [ ]:
# read a few diagnoses
recs = [json.loads(l) for l in open('outputs/explanations/explanations_hf.jsonl')]
for r in recs[:3]:
    print('=' * 80); print('Student', r['user']); print(r['text'])
    issues = r['faithfulness']['issues']
    print('Faithful' if not issues else f'Unfaithful: {issues}')

## 8. (Optional) Claude for comparison

Uses your Anthropic API key, typed in privately so it is never saved in the notebook.
Running both lets you compare an open model with a hosted one, as in the proposal.

In [ ]:
from getpass import getpass
os.environ['ANTHROPIC_API_KEY'] = getpass('Anthropic API key: ')
!python -m genex.explain --backend claude --n 100 2>&1 | grep -v -i warn | tail -2
!python -m genex.intervene --backend claude 2>&1 | grep -v -i warn | tail -4

In [ ]:
# side-by-side comparison of the explainers (same 100 students)
rows = []
for b in ['template', 'hf', 'claude']:
    p = f'outputs/explanations/summary_{b}.json'
    if os.path.exists(p):
        rows.append(json.load(open(p)))
pd.DataFrame(rows)[['backend', 'n', 'faithfulness_pass_rate', 'claim_level_faithfulness',
                    'future_acc_on_flagged_gaps', 'future_acc_on_flagged_mastered']].round(3)

## 9. Evaluation report and expert sheets

Builds the prediction table, DeLong tests and a blinded rating sheet per explainer in
`outputs/report/`. Intervention sheets are in `outputs/interventions/`; keep the `_KEY`
files away from the raters.

In [ ]:
!python -m genex.evaluate report 2>&1 | grep -v -i warn
!ls outputs/report outputs/interventions

## 10. (Optional) Full-size backbone on the GPU

The included results used a small model on CPU. On a GPU you can tune and retrain at full size.
The `--trials 30` search takes a few hours on a T4; use the proposal's 100 trials if you have
Colab Pro. Rerun steps 6–9 afterwards so explanations use the new model.

In [ ]:
# hyper-parameter search (resumable: progress is saved in outputs/tuning/optuna.db)
!python -m genex.tune --trials 30 --epochs 15

In [ ]:
# retrain all models; GENEX-KT uses the tuned settings
!python run_all.py --params outputs/tuning/best_params.json --backend hf

## 11. Launch the demo app

Starts the GENEX-KT web app on this Colab machine and shows it below. Pick a student or
upload a CSV. It uses the open-source model from step 4 as an explainer (choose `hf` in
the *Explainer* menu; the first call loads the model, about 1–2 minutes), the offline
`template` explainer, or Claude if you set a key in step 8. The app stops when the Colab
session ends; rerun this cell to start it again.

In [ ]:
import subprocess, time, urllib.request
!pip install -q gradio plotly
env = dict(os.environ, GENEX_ENABLE_HF='1', PYTHONWARNINGS='ignore')
if os.path.exists('outputs_gpu/genex/model.pt'):
    env['GENEX_OUT_DIR'] = 'outputs_gpu'
app = subprocess.Popen(['python', 'app.py'], env=env,
                       stdout=open('/content/app.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        urllib.request.urlopen('http://127.0.0.1:7860/'); break
    except Exception:
        time.sleep(2)
from google.colab import output
output.serve_kernel_port_as_iframe(7860, height=1400)

## 12. Download everything

In [ ]:
!cd /content/drive/MyDrive && zip -qr genex_outputs.zip genex_kt/outputs
from google.colab import files
files.download('/content/drive/MyDrive/genex_outputs.zip')